# 32 - Immersion Obstruction Theory: Stiefel-Whitney, Pontryagin & Maslov

Whitney's theorem says every smooth $n$-manifold immerses in $\mathbb{R}^{2n}$. The question is whether you can do better: can $M^n$ immerse in $\mathbb{R}^{n+k}$ for $k < n$?

The answer is encoded in **characteristic classes**:
- **Dual Stiefel-Whitney classes** $\bar{w}_i(M)$: if $M \looparrowright \mathbb{R}^{n+k}$, then $\bar{w}_i = 0$ for $i > k$
- **Pontryagin classes** $p_i(M)$: rational obstructions to immersion in lower-dimensional Euclidean spaces
- **Euler class** $e(M)$: top-dimensional obstruction to a non-vanishing section

This notebook covers `pysurgery.core.immersion_obstructions` systematically. For the Whitney trick (removing self-intersections of *existing* immersions), see NB 17.

## Learning Goals
- Compute dual Stiefel-Whitney classes with `compute_dual_stiefel_whitney_classes`
- Certify non-immersibility with `check_dual_stiefel_whitney_non_immersibility`
- Compute Pontryagin and Euler classes
- Run a full analysis with `immersion_obstruction_analysis`
- Understand `StructuralObstruction`, `NonImmersibilityWitness`, `ImmersibilityInconclusive`

## Three-Level View
| Level | Object | Tool |
|---|---|---|
| **Geometric** | Smooth $n$-manifold $M$ | `SimplicialComplex` |
| **Algebraic** | Characteristic classes $\bar{w}_i \in H^i(M; \mathbb{Z}/2)$ | `compute_dual_stiefel_whitney_classes` |
| **Result** | Non-immersibility witness or inconclusiveness | `NonImmersibilityWitness` / `ImmersibilityInconclusive` |

## Formal Background

The **dual Stiefel-Whitney classes** are defined by $w(TM) \smile \bar{w}(TM) = 1$ (the total class). Whitney's formula: if $M \looparrowright \mathbb{R}^{n+k}$, then $\bar{w}(TM) = w(\nu)$ where $\nu$ is the normal bundle of rank $k$. Since $\nu$ is a rank-$k$ bundle, $w_i(\nu) = 0$ for $i > k$, giving $\bar{w}_i(M) = 0$ for $i > k$.

In [ ]:
import numpy as np
from pysurgery.core.immersion_obstructions import (
    compute_dual_stiefel_whitney_classes,
    check_dual_stiefel_whitney_non_immersibility,
    rational_pontryagin_classes, PontryaginClasses,
    combinatorial_euler_class, EulerClass,
    compute_rational_pontryagin_obstruction,
    immersion_obstruction_analysis,
    NonImmersibilityWitness,
)
from pysurgery.core.complexes import SimplicialComplex

print("=" * 70)
print("32 - Immersion Obstruction Theory: Setup Complete")
print("=" * 70)

## Part 1: Dual Stiefel-Whitney Classes

The dual SW class $\bar{w}_k(M)$ lives in $H^k(M; \mathbb{Z}/2)$. It is computed combinatorially from the simplicial structure via the total Stiefel-Whitney class of the tangent bundle and the relation $w \smile \bar{w} = 1$.

Non-zero $\bar{w}_k$ means: any immersion of $M$ in $\mathbb{R}^{n+j}$ requires $j \geq k$.

In [ ]:
# RP^n are the standard test cases for Stiefel-Whitney obstructions
rp_spaces = {
    "RP²": SimplicialComplex.real_projective_space(2),
    "RP³": SimplicialComplex.real_projective_space(3),
    "RP⁴": SimplicialComplex.real_projective_space(4),
}

for name, M in rp_spaces.items():
    dual_sw = compute_dual_stiefel_whitney_classes(M)
    nonzero = [(i, w) for i, w in enumerate(dual_sw) if np.any(w) and i > 0]
    n = M.dimension
    if nonzero:
        top_obs = max(i for i, _ in nonzero)
        print(f"{name} (dim={n}): non-zero w̄_i for i ∈ {[i for i, _ in nonzero]}")
        print(f"  → requires immersion codimension ≥ {top_obs}")
    else:
        print(f"{name}: all dual SW classes vanish (no SW obstruction)")

## Part 2: Non-Immersibility Witnesses

`check_dual_stiefel_whitney_non_immersibility` checks whether $\bar{w}_i \neq 0$ for some $i > k$, where $k = \text{target\_dim} - n$ is the codimension. Returns:
- `NonImmersibilityWitness`: obstruction found, immersion in target dimension is impossible
- `ImmersibilityInconclusive`: no SW obstruction; immersion *may* exist (need other methods)
- `StructuralObstruction`: input data is ill-formed

In [ ]:
# Systematic check for RP⁴
rp4 = SimplicialComplex.real_projective_space(4)
print("Immersibility check for RP⁴ (dim=4):")
print()

for target in range(5, 10):
    result = check_dual_stiefel_whitney_non_immersibility(rp4, target_dim=target)
    rtype = type(result).__name__
    details = ""
    if isinstance(result, NonImmersibilityWitness):
        details = f"← w̄_{result.dimension_blocked} ≠ 0"
    print(f"  R^{target}: {rtype:35s} {details}")

print()
print("RP⁴ → R^7 is first codimension where SW obstruction vanishes")
print("(RP⁴ indeed immerses in R^7; Haefliger, 1961)")

## Part 3: Pontryagin Classes (Rational Obstructions)

Over $\mathbb{Q}$, the tangent bundle is classified by **Pontryagin classes** $p_i(M) \in H^{4i}(M; \mathbb{Z})$. These give rational obstructions: if $M \looparrowright \mathbb{R}^{n+k}$, then $p_i(M) = 0$ for $4i > k$. Unlike SW classes, Pontryagin classes are sensitive to orientation and not just mod-2 topology.

In [ ]:
# Pontryagin classes for CP²
CP2 = SimplicialComplex.complex_projective_space(2)
pc: PontryaginClasses = rational_pontryagin_classes(CP2)

print("Pontryagin classes for CP² (dim=4):")
print(f"  p₁ = {pc.p1}  (expected 3 for CP²)")
print(f"  p₂ = {pc.p2}  (not defined for dim < 8, 0 here)")
print(f"  exact: {pc.exact}")
print(f"  theorem_tag: {pc.theorem_tag}")
print()

# Pontryagin obstruction check
obs_rat = compute_rational_pontryagin_obstruction(CP2, target_dim=5)
print(f"Rational Pontryagin obstruction for CP² → R^5: {type(obs_rat).__name__}")
if hasattr(obs_rat, 'obstruction_class'):
    print(f"  obstruction_class: {obs_rat.obstruction_class}")

## Part 4: Euler Class

The **Euler class** $e(E) \in H^n(M; \mathbb{Z})$ of a rank-$n$ bundle $E$ is the top obstruction to a non-vanishing section. For the tangent bundle, $e(TM)$ is Poincaré dual to the zero set of a generic vector field, i.e., $e(TM) = \chi(M) \cdot [\text{pt}]$ (Poincaré-Hopf).

Non-zero Euler class implies no non-vanishing tangent vector field, and is an obstruction to certain immersions.

In [ ]:
# Euler class for even-dimensional manifolds
for name, M in [("S²", SimplicialComplex.sphere(2)),
                ("T²", SimplicialComplex.torus()),
                ("CP²", SimplicialComplex.complex_projective_space(2))]:
    e: EulerClass = combinatorial_euler_class(M)
    print(f"{name}: e = {e.euler_number}  (= χ = Euler characteristic)")
    print(f"  exact: {e.exact}")

## Part 5: Full Immersion Obstruction Analysis

`immersion_obstruction_analysis` combines all three obstructions (SW, Pontryagin, Euler) into a single comprehensive report.

In [ ]:
# Full analysis for RP⁴ → R^7
rp4 = SimplicialComplex.real_projective_space(4)
analysis = immersion_obstruction_analysis(rp4, target_dim=7)

print("Full immersion obstruction analysis: RP⁴ → R^7")
print(f"  can_immerse:    {analysis.can_immerse}")
print(f"  obstructions:   {analysis.obstructions}")
print(f"  sw_result:      {type(analysis.sw_result).__name__}")
print(f"  pontryagin_obs: {analysis.pontryagin_obstruction}")
print(f"  euler_obs:      {analysis.euler_obstruction}")
print(f"  exact:          {analysis.exact}")
print(f"  theorem_tag:    {analysis.theorem_tag}")
print()

# Compare: RP⁴ → R^6 (should be obstructed)
analysis_6 = immersion_obstruction_analysis(rp4, target_dim=6)
print(f"RP⁴ → R^6: can_immerse={analysis_6.can_immerse}")
print(f"  obstructions: {analysis_6.obstructions}")

## Summary Checklist

- [x] Computed dual Stiefel-Whitney classes with `compute_dual_stiefel_whitney_classes`
- [x] Used `check_dual_stiefel_whitney_non_immersibility` and read all three result types
- [x] Computed Pontryagin classes with `rational_pontryagin_classes`
- [x] Computed Euler class and connected to Euler characteristic
- [x] Used `immersion_obstruction_analysis` for a full combined report

## Next Steps
- **NB 17**: Whitney trick — removing self-intersections of existing immersions
- **NB 25**: Poincaré duality — Euler class appears in the cap product framework
- **NB 34**: The capstone uses immersion obstructions in the embedding step